In [0]:
import os, sys, time
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, bronze

cfg = common.load_config("dev")
key = common.match_key("dev")
table, spec = next((t, s) for t, s in bronze.load_tables().items()
                   if s["format"] == "csv" and not s.get("drift"))
base = f"/Volumes/{cfg['catalog']}/bronze"
folder = f"{spec['source']}/{table}/2026/10"
bad = {f"{table}_20261006_09.csv": "a;b;c\n1;2;3\n",      # wrong delimiter
       f"{table}_latest.csv": "a,b\n1,2\n"}                 # wrong file name
os.makedirs(f"{base}/landing/{folder}", exist_ok=True)
for name, body in bad.items():
    with open(f"{base}/landing/{folder}/{name}", "w") as f:
        f.write(body)

start = time.time()
result = bronze.load_table(spark, cfg, table, spec, key, bronze.list_landing(spark, cfg))
print(table, "(rows, loaded, rejected) =", result, f"in {time.time() - start:.0f}s")

found = [n for n in bad if os.path.exists(f"{base}/rejected/{folder}/{n}")]
for n in found:
    os.remove(f"{base}/rejected/{folder}/{n}")              # remove the test files
for stage in ("landing", "rejected"):                       # remove the empty test folders
    try:
        os.removedirs(f"{base}/{stage}/{folder}")
    except OSError:
        pass
print("PASS" if result == (0, 0, 2) and len(found) == 2 else "FAIL")